# **Clinical Trial Enrichment: Run 4 (Structural Anchor)**
---
This notebook orchestrates the fourth stage of the clinical enrichment pipeline. While previous runs focused on *what* disease is being treated (Run 1), *what* drug is being used (Run 2), and *how* the trial is designed (Run 3), Run 4 identifies the **Structural Anchors**: the standardized **Lead Sponsor** (Parent Resolution) and the **Primary Operational Duration**.

It implements the **'Safe-Duration' Strategy (v18.1)**: harvesting raw endpoint data with a 3,000-character safety limit to ensure that long-term primary outcomes (e.g., Overall Survival) are not truncated, while maintaining a lean context for token efficiency.

# **0. Master Control: Reset & Configuration**
Define the execution mode, sampling parameters, and reset artifacts upfront. This ensures reproducibility and prevents the accidental merging of stale data.

In [1]:
# [CONTROL] Set to True to enable deletion of existing input contexts
RESET_PRODUCED_FILES = True

# [CONFIG] Toggle between full run and test sample
IS_PRODUCTION = True  # Set to True for full run
SAMPLE_SIZE = 2000
RANDOM_STATE = 2026

import os
files_to_delete = [
        '../data/llm_in_04.csv',
    ]

if RESET_PRODUCED_FILES:
    for f in files_to_delete:
        if os.path.exists(f):
            os.remove(f)
            print(f"> Deleted: {f}")
    print("> Reset complete.")
else:
    print("> Reset skipped.")

> Reset complete.


# **1. Environment Setup & Configuration**
Initialize libraries and project-specific utilities. We import the `day_zero_reconstructor` to sanitize linguistic context and define path constants.

In [2]:
import pandas as pd
import os
import csv
import re
import sys
import json
from collections import defaultdict
from dotenv import load_dotenv

load_dotenv()
sys.path.append('..')
from src.prep.text_cleaning import day_zero_reconstructor

DATA_PATH = '../data/'
OUTPUT_PATH = '../data/processed'
NL = chr(10)

# [STEP 5] Utility function for robust CSV loading with specific clinical formatting
def safe_load(filename, cols=None):
    full_path = os.path.join(DATA_PATH, filename)
    # Match Strategy A: PERFECT from data_loader_clinpred.py
    params_perfect = {
        "sep": "|", "dtype": str, "header": 0, "quotechar": '"',
        "quoting": csv.QUOTE_MINIMAL, "low_memory": False, "on_bad_lines": "warn"
    }
    # Match Strategy B: ROBUST
    params_robust = {
        "sep": "|", "dtype": str, "header": 0, "quotechar": '"',
        "quoting": 3, "low_memory": False, "on_bad_lines": "warn"
    }
    try:
        return pd.read_csv(full_path, usecols=cols, **params_perfect)
    except:
        return pd.read_csv(full_path, usecols=cols, **params_robust)

print("> SUCCESS: Environment Ready.")


> SUCCESS: Environment Ready.


# **2. Anchor Alignment & Evidence Harvesting**
We harvest core metadata (Title, Year), Lead Sponsors, and Primary Outcomes directly from the raw AACT source files. We also inherit the `alpha_drug_name` from Run 2 to provide the agent context.

In [3]:
print(">>> Harvesting Evidence for Run 4...")

# [STEP 1] Load Target Cohort (from Run 1 results)
df_target = pd.read_csv(os.path.join(OUTPUT_PATH, 'llm_out_01.csv'), usecols=['nct_id'])
if not IS_PRODUCTION:
    df_target = df_target.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)
target_ids = df_target['nct_id'].tolist()

# [STEP 2] Load Run 2 Drug Anchors
df_run2 = pd.read_csv(os.path.join(OUTPUT_PATH, 'llm_out_02.csv'), usecols=['nct_id', 'alpha_drug_name'])
run2_lookup = df_run2[df_run2['nct_id'].isin(target_ids)].set_index('nct_id').to_dict('index')

# [STEP 3] Study Metadata (Title, Year)
df_studies = safe_load('studies.txt', cols=['nct_id', 'official_title', 'brief_title', 'start_date'])
df_studies['official_title'] = df_studies['official_title'].fillna(df_studies['brief_title'])
studies_lookup = df_studies[df_studies['nct_id'].isin(target_ids)].set_index('nct_id').to_dict('index')

# [STEP 4] Lead Sponsors
df_sponsors = safe_load('sponsors.txt', cols=['nct_id', 'lead_or_collaborator', 'name'])
leads = df_sponsors[(df_sponsors['lead_or_collaborator'].str.lower() == 'lead') & (df_sponsors['nct_id'].isin(target_ids))]
sponsor_lookup = {row['nct_id']: row['name'] for _, row in leads.iterrows()}

# [STEP 5] Primary Outcomes (The 3,000 Chars Safe-Duration Rule)
df_outcomes = safe_load('design_outcomes.txt', cols=['nct_id', 'outcome_type', 'measure', 'time_frame'])
primary_outcomes_lookup = defaultdict(list)
for _, row in df_outcomes[df_outcomes['nct_id'].isin(target_ids)].iterrows():
    if str(row['outcome_type']).strip().upper() == 'PRIMARY':
        m = day_zero_reconstructor(str(row['measure']), "outcome")[:500] # Smart line-level cap
        t = day_zero_reconstructor(str(row['time_frame']), "timeframe")
        primary_outcomes_lookup[row['nct_id']].append(f"TITLE: {m} | TIMEFRAME: {t}")

print(f"> Harvesting Complete for {len(target_ids)} trials.")

>>> Harvesting Evidence for Run 4...
> Harvesting Complete for 29557 trials.


# **3. Context Assembly & Linguistic Diet Enforcement**
The final step iterates through the cohort and assembles the specialized Run 4 context. We enforce the **3,000-character limit** for endpoints to ensure no duration data is lost.

In [4]:
results = []
for nct_id in target_ids:
    s = studies_lookup.get(nct_id, {})
    r2 = run2_lookup.get(nct_id, {})

    # Base Metadata
    start_date = s.get('start_date', '2000-01-01')
    start_year = str(pd.to_datetime(start_date).year)
    title = day_zero_reconstructor(s.get('official_title', 'Unknown'), "title")[:500]
    agent = r2.get('alpha_drug_name', 'Unknown')
    sponsor = sponsor_lookup.get(nct_id, 'Unknown')

    # Safe Endpoints (3,000 char cap)
    outcomes = primary_outcomes_lookup.get(nct_id, ["No primary outcome listed"])
    # Join with separators and apply the safety limit
    outcome_str = (NL + "---" + NL).join(outcomes)[:3000]

    # Assemble Refined Run 4 Context
    # [STRATEGY: REMOVED ALL BEHAVIORAL TAGS TO SAVE TOKENS]
    context_body = (
        f"[NCT_ID]: {nct_id}{NL}"
        f"[START_YEAR]: {start_year}{NL}"
        f"[OFFICIAL_TITLE]: {title}{NL}"
        f"[RAW_SPONSOR]: {sponsor}{NL}"
        f"[AGENT]: {agent}{NL}"
        f"{NL}[PRIMARY_ENDPOINTS]:{NL}{outcome_str}"
    )

    results.append({"nct_id": nct_id, "context": context_body})

# Export to llm_in_04.csv
pd.DataFrame(results).to_csv(os.path.join(DATA_PATH, 'llm_in_04.csv'), index=False)
print(f"> Success: Created data/llm_in_04.csv with {len(results)} trials.")

> Success: Created data/llm_in_04.csv with 29557 trials.


---
## **Next Step: Run Structural Anchor Orchestrator**
After generating the context CSV in this notebook, shift to your terminal and execute the following command to start the Stage 4 extraction:

```bash
python3 src/prep/llm_in_04_run.py
```